# Predictive Maintenance — AI4I 2020 Dataset

**Proyek Machine Learning: Klasifikasi Kegagalan Mesin**

Dataset: `ai4i2020.csv` | 10.000 baris × 14 kolom

---
| Step | Deskripsi |
|---|---|
| **Step 1** | Persiapan Data & EDA |
| **Step 2** | Training & Testing Model Supervised Learning |
| **Step 3** | Evaluasi & Pemilihan Model Terbaik |

## Instalasi & Import Library

In [ ]:
# Install library yang dibutuhkan (jalankan di Google Colab)
!pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn joblib -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
 accuracy_score, precision_score, recall_score,
 f1_score, confusion_matrix, classification_report
)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(' Semua library berhasil di-import!')

---
# STEP 1: Persiapan Data & EDA (Exploratory Data Analysis)

## 1.1 Load Dataset & Informasi Awal

In [ ]:
# Jika menggunakan Google Colab, upload file csv terlebih dahulu:
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv('ai4i2020.csv')

print('=' * 60)
print(' INFORMASI DATASET')
print('=' * 60)
print(f'Shape : {df.shape}')
print(f'Jumlah Baris : {df.shape[0]:,}')
print(f'Jumlah Kolom : {df.shape[1]}')
print()
print(' 5 Baris Pertama:')
df.head()

In [ ]:
print(' Tipe Data Setiap Kolom:')
print(df.dtypes)
print()
print(' Statistik Deskriptif:')
df.describe().round(2)

## 1.2 Pengecekan & Penanganan Missing Values

In [ ]:
print('=' * 60)
print(' PENGECEKAN MISSING VALUES')
print('=' * 60)

missing = df.isnull().sum()
print('Jumlah Missing Values per Kolom:')
print(missing)
print(f'\nTotal keseluruhan missing values: {missing.sum()}')

# Visualisasi heatmap missing values
plt.figure(figsize=(14, 3))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Heatmap Missing Values', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

if missing.sum() == 0:
 print('\n Tidak ada missing values! Dataset sudah bersih.')
else:
 df.dropna(inplace=True)
 print(f'\n Missing values dihapus. Shape baru: {df.shape}')

## 1.3 Pengecekan & Penanganan Data Duplikat

In [ ]:
print('=' * 60)
print(' PENGECEKAN DATA DUPLIKAT')
print('=' * 60)

n_dup = df.duplicated().sum()
print(f'Jumlah baris duplikat: {n_dup}')

if n_dup > 0:
 df.drop_duplicates(inplace=True)
 print(f' Duplikat dihapus. Shape baru: {df.shape}')
else:
 print(' Tidak ada data duplikat!')

## 1.4 Deteksi & Penanganan Outlier (Metode IQR)

In [ ]:
print('=' * 60)
print(' DETEKSI OUTLIER — IQR METHOD')
print('=' * 60)

numeric_cols = [
 'Air temperature [K]',
 'Process temperature [K]',
 'Rotational speed [rpm]',
 'Torque [Nm]',
 'Tool wear [min]'
]

# Boxplot SEBELUM penanganan
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
fig.suptitle('Boxplot SEBELUM Penanganan Outlier', fontsize=14, fontweight='bold', color='red')
for i, col in enumerate(numeric_cols):
 axes[i].boxplot(df[col].dropna(), patch_artist=True,
 boxprops=dict(facecolor='#EF9A9A', color='#C62828'),
 medianprops=dict(color='#C62828', linewidth=2))
 short = col.split('[')[0].strip()
 axes[i].set_title(short, fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

# Hitung jumlah outlier per kolom
print('\n Jumlah Outlier per Kolom (IQR Method):')
outlier_info = []
for col in numeric_cols:
 Q1 = df[col].quantile(0.25)
 Q3 = df[col].quantile(0.75)
 IQR = Q3 - Q1
 lower = Q1 - 1.5 * IQR
 upper = Q3 + 1.5 * IQR
 n_out = df[(df[col] < lower) | (df[col] > upper)].shape[0]
 outlier_info.append({'Kolom': col, 'Q1': round(Q1,3), 'Q3': round(Q3,3),
 'IQR': round(IQR,3), 'Lower Bound': round(lower,3),
 'Upper Bound': round(upper,3), 'Jumlah Outlier': n_out})
 print(f' {col}: {n_out} outlier')

pd.DataFrame(outlier_info)

In [ ]:
# Penanganan outlier dengan CLIPPING (batas IQR)
df_clean = df.copy()

for col in numeric_cols:
 Q1 = df_clean[col].quantile(0.25)
 Q3 = df_clean[col].quantile(0.75)
 IQR = Q3 - Q1
 lower = Q1 - 1.5 * IQR
 upper = Q3 + 1.5 * IQR
 df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)

# Boxplot SESUDAH penanganan
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
fig.suptitle('Boxplot SESUDAH Penanganan Outlier (Clipping IQR)', fontsize=14, fontweight='bold', color='green')
for i, col in enumerate(numeric_cols):
 axes[i].boxplot(df_clean[col].dropna(), patch_artist=True,
 boxprops=dict(facecolor='#A5D6A7', color='#1B5E20'),
 medianprops=dict(color='#1B5E20', linewidth=2))
 short = col.split('[')[0].strip()
 axes[i].set_title(short, fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

print(f' Outlier berhasil ditangani dengan metode Clipping IQR!')
print(f'Shape setelah penanganan outlier: {df_clean.shape}')

## 1.5 Feature Engineering

In [ ]:
print('=' * 60)
print(' FEATURE ENGINEERING')
print('=' * 60)

# 1. Selisih suhu (domain knowledge: HDF terjadi jika diff < 8.6 K)
df_clean['Temp_diff'] = (
 df_clean['Process temperature [K]'] - df_clean['Air temperature [K]']
)

# 2. Power (kW) = Torque x RPM / 9550 (rumus fisika mekanik)
df_clean['Power'] = (
 df_clean['Torque [Nm]'] * df_clean['Rotational speed [rpm]'] / 9550
)

# 3. Torque x Tool Wear (indikator overstrain)
df_clean['Torque_x_ToolWear'] = (
 df_clean['Torque [Nm]'] * df_clean['Tool wear [min]']
)

# 4. Encoding kolom 'Type' dengan Label Encoding
le = LabelEncoder()
df_clean['Type_encoded'] = le.fit_transform(df_clean['Type'])
encoding_map = dict(zip(le.classes_, le.transform(le.classes_)))
print(f'Encoding Type: {encoding_map}')

# 5. Drop kolom ID dan kolom Type asli
df_clean.drop(columns=['UDI', 'Product ID', 'Type'], inplace=True)

print('\n Fitur baru yang ditambahkan:')
print(' - Temp_diff : Process temp - Air temp [K]')
print(' - Power : Torque x RPM / 9550 [kW]')
print(' - Torque_x_ToolWear: Torque x Tool wear')
print(' - Type_encoded : Label Encoding dari kolom Type')
print(f'\nShape setelah Feature Engineering: {df_clean.shape}')
print(f'\nKolom-kolom sekarang:')
print(list(df_clean.columns))

## 1.6 Normalisasi / Standarisasi Fitur (StandardScaler)

In [ ]:
print('=' * 60)
print(' NORMALISASI — StandardScaler')
print('=' * 60)

target_cols = ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
feature_cols = [c for c in df_clean.columns if c not in target_cols]

print(f'Kolom yang dinormalisasi: {feature_cols}')

scaler = StandardScaler()
df_clean[feature_cols] = scaler.fit_transform(df_clean[feature_cols])

print('\n Normalisasi selesai! (mean ≈ 0, std ≈ 1)')
print('\nStatistik fitur setelah normalisasi:')
df_clean[feature_cols].describe().round(3)

## 1.7 Visualisasi Data

In [ ]:
# HISTOGRAM semua fitur 
print(' Histogram Distribusi Semua Fitur (Setelah Normalisasi)')

fig, axes = plt.subplots(3, 4, figsize=(22, 14))
fig.suptitle('Distribusi Semua Fitur (Setelah Normalisasi & Feature Engineering)',
 fontsize=15, fontweight='bold', y=1.02)
axes = axes.flatten()

colors = plt.cm.Set2(np.linspace(0, 1, len(feature_cols)))
for i, col in enumerate(feature_cols):
 axes[i].hist(df_clean[col], bins=40, color=colors[i], edgecolor='white', alpha=0.85)
 axes[i].set_title(col, fontsize=9, fontweight='bold')
 axes[i].set_xlabel('Nilai (Normalized)', fontsize=8)
 axes[i].set_ylabel('Frekuensi', fontsize=8)

for j in range(len(feature_cols), len(axes)):
 axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# CORRELATION MATRIX 
print(' Correlation Matrix')

corr_cols = feature_cols + ['Machine failure']
corr = df_clean[corr_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(14, 11))
sns.heatmap(
 corr, mask=mask, annot=True, fmt='.2f',
 cmap='RdYlGn', center=0, square=True,
 linewidths=0.5, cbar_kws={'shrink': 0.75},
 annot_kws={'size': 8}
)
plt.title('Correlation Matrix — Fitur & Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# DISTRIBUSI KELAS & MODE KEGAGALAN 
print(' Distribusi Kelas Target & Mode Kegagalan')

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Analisis Distribusi Kelas', fontsize=15, fontweight='bold')

# 1. Bar chart Machine failure
fc = df_clean['Machine failure'].value_counts().sort_index()
bars = axes[0].bar(['Normal (0)', 'Failure (1)'], fc.values,
 color=['#1976D2', '#D32F2F'], edgecolor='white', width=0.5)
axes[0].set_title('Distribusi Machine Failure', fontweight='bold', fontsize=11)
axes[0].set_ylabel('Jumlah Data')
for bar, val in zip(bars, fc.values):
 axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
 f'{val}\n({val/len(df_clean)*100:.1f}%)',
 ha='center', fontweight='bold', fontsize=10)

# 2. Bar chart mode kegagalan
failure_modes = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
failure_labels = ['Tool Wear\nFailure', 'Heat Dissipation\nFailure',
 'Power\nFailure', 'Overstrain\nFailure', 'Random\nFailure']
fm_counts = [df_clean[m].sum() for m in failure_modes]
mode_colors = ['#FF9800', '#9C27B0', '#00BCD4', '#4CAF50', '#FF5722']
bars2 = axes[1].bar(failure_labels, fm_counts, color=mode_colors, edgecolor='white')
axes[1].set_title('Distribusi Mode Kegagalan', fontweight='bold', fontsize=11)
axes[1].set_ylabel('Jumlah Kasus')
for bar, val in zip(bars2, fm_counts):
 axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
 str(val), ha='center', fontweight='bold', fontsize=10)

# 3. Pie chart tipe mesin (dari data asli)
type_counts_val = [6000, 2997, 1003]
type_labels_pie = ['L — Low Quality\n(60%)', 'M — Medium Quality\n(30%)', 'H — High Quality\n(10%)']
axes[2].pie(type_counts_val, labels=type_labels_pie, autopct='%1.1f%%',
 colors=['#42A5F5', '#66BB6A', '#FFA726'],
 startangle=90, explode=(0.05, 0.05, 0.05),
 textprops={'fontsize': 9})
axes[2].set_title('Distribusi Tipe Mesin', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

print('\n Semua visualisasi Step 1 selesai!')

---
# STEP 2: Training & Testing Model Supervised Learning

## 2.1 Definisi Fitur & Target

In [ ]:
# Fitur (X) dan target (y)
X = df_clean[feature_cols].copy()
y = df_clean['Machine failure'].copy()

print(' Fitur (X):')
print(list(X.columns))
print(f'\nShape X: {X.shape}')
print(f'\nDistribusi Target (y = Machine failure):')
print(y.value_counts())
print(f'\nRasio kelas:')
print(y.value_counts(normalize=True).round(4))
print('\n Dataset IMBALANCED! Kelas failure hanya ~3.39%')
print(' → Akan ditangani menggunakan SMOTE pada training set.')

## 2.2 Splitting Data Train / Test (80:20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
 X, y,
 test_size=0.2,
 random_state=42,
 stratify=y # Menjaga proporsi kelas failure di kedua set
)

print('=' * 60)
print(' SPLITTING DATA 80:20')
print('=' * 60)
print(f'Total data : {len(X):,} sampel')
print(f'Training set : {len(X_train):,} sampel (80%)')
print(f'Testing set : {len(X_test):,} sampel (20%)')
print(f'\nDistribusi kelas di Training set:')
print(y_train.value_counts())
print(f'\nDistribusi kelas di Testing set:')
print(y_test.value_counts())

# Handle imbalanced dengan SMOTE (HANYA pada training set)
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'\n Setelah SMOTE (Training set saja):')
print(pd.Series(y_train_sm).value_counts())
print(f'Training set setelah SMOTE: {len(X_train_sm):,} sampel')
print('\n Data siap untuk training!')

## 2.3 Model 1 — Random Forest Classifier

In [ ]:
print('=' * 60)
print(' MODEL 1: RANDOM FOREST CLASSIFIER')
print('=' * 60)

rf_model = RandomForestClassifier(
 n_estimators=100,
 max_depth=None,
 min_samples_split=2,
 min_samples_leaf=1,
 random_state=42,
 n_jobs=-1
)

rf_model.fit(X_train_sm, y_train_sm)

print(' Random Forest berhasil dilatih!')
print(f' n_estimators : 100 pohon')
print(f' max_depth : None (tidak dibatasi)')
print(f' random_state : 42')
print(f' Jumlah fitur : {X_train_sm.shape[1]}')
print(f' Jumlah sampel : {X_train_sm.shape[0]:,}')

## 2.4 Model 2 — Decision Tree Classifier

In [ ]:
print('=' * 60)
print(' MODEL 2: DECISION TREE CLASSIFIER')
print('=' * 60)

dt_model = DecisionTreeClassifier(
 max_depth=10,
 min_samples_split=5,
 min_samples_leaf=2,
 random_state=42
)

dt_model.fit(X_train_sm, y_train_sm)

print(' Decision Tree berhasil dilatih!')
print(f' max_depth : 10')
print(f' min_samples_split: 5')
print(f' random_state : 42')
print(f' Jumlah fitur : {X_train_sm.shape[1]}')
print(f' Jumlah sampel : {X_train_sm.shape[0]:,}')

---
# STEP 3: Evaluasi Model

## 3.1 Evaluasi Model 1 — Random Forest

In [ ]:
print('=' * 60)
print(' Evaluasi MODEL 1: RANDOM FOREST')
print('=' * 60)

y_pred_rf = rf_model.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf, zero_division=0)
rec_rf = recall_score(y_test, y_pred_rf, zero_division=0)
f1_rf = f1_score(y_test, y_pred_rf, zero_division=0)

print(f'Akurasi : {acc_rf:.4f} ({acc_rf*100:.2f}%)')
print(f'Presisi : {prec_rf:.4f}')
print(f'Recall : {rec_rf:.4f}')
print(f'F1-Score : {f1_rf:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['Normal', 'Failure']))

# Confusion Matrix RF
cm_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
 xticklabels=['Normal', 'Failure'],
 yticklabels=['Normal', 'Failure'],
 linewidths=1, linecolor='white', annot_kws={'size': 14})
plt.title('Confusion Matrix — Random Forest', fontsize=13, fontweight='bold')
plt.ylabel('Aktual', fontsize=12)
plt.xlabel('Prediksi', fontsize=12)
plt.tight_layout()
plt.show()

## 3.2 Evaluasi Model 2 — Decision Tree

In [ ]:
print('=' * 60)
print(' Evaluasi MODEL 2: DECISION TREE')
print('=' * 60)

y_pred_dt = dt_model.predict(X_test)

acc_dt = accuracy_score(y_test, y_pred_dt)
prec_dt = precision_score(y_test, y_pred_dt, zero_division=0)
rec_dt = recall_score(y_test, y_pred_dt, zero_division=0)
f1_dt = f1_score(y_test, y_pred_dt, zero_division=0)

print(f'Akurasi : {acc_dt:.4f} ({acc_dt*100:.2f}%)')
print(f'Presisi : {prec_dt:.4f}')
print(f'Recall : {rec_dt:.4f}')
print(f'F1-Score : {f1_dt:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_dt, target_names=['Normal', 'Failure']))

# Confusion Matrix DT
cm_dt = confusion_matrix(y_test, y_pred_dt)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Oranges',
 xticklabels=['Normal', 'Failure'],
 yticklabels=['Normal', 'Failure'],
 linewidths=1, linecolor='white', annot_kws={'size': 14})
plt.title('Confusion Matrix — Decision Tree', fontsize=13, fontweight='bold')
plt.ylabel('Aktual', fontsize=12)
plt.xlabel('Prediksi', fontsize=12)
plt.tight_layout()
plt.show()

## 3.3 Perbandingan Kedua Model & Pemilihan Model Terbaik

In [ ]:
print('=' * 60)
print(' TABEL PERBANDINGAN KEDUA MODEL')
print('=' * 60)

results = pd.DataFrame({
 'Model' : ['Random Forest', 'Decision Tree'],
 'Akurasi' : [acc_rf, acc_dt],
 'Presisi' : [prec_rf, prec_dt],
 'Recall' : [rec_rf, rec_dt],
 'F1-Score' : [f1_rf, f1_dt]
})

results_display = results.copy()
for col in ['Akurasi', 'Presisi', 'Recall', 'F1-Score']:
 results_display[col] = results_display[col].map(lambda x: f'{x:.4f}')

print(results_display.to_string(index=False))

# Visualisasi perbandingan 
metrics = ['Akurasi', 'Presisi', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.32

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar chart
bars1 = axes[0].bar(x - width/2, results.iloc[0][metrics], width,
 label=' Random Forest', color='#1565C0', alpha=0.87, edgecolor='white')
bars2 = axes[0].bar(x + width/2, results.iloc[1][metrics], width,
 label=' Decision Tree', color='#E65100', alpha=0.87, edgecolor='white')
axes[0].set_xlabel('Metrik Evaluasi', fontsize=12)
axes[0].set_ylabel('Nilai', fontsize=12)
axes[0].set_title('Perbandingan Performa Model', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics, fontsize=11)
axes[0].set_ylim(0, 1.2)
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', alpha=0.3)
for bar in bars1:
 h = bar.get_height()
 axes[0].text(bar.get_x()+bar.get_width()/2, h+0.01, f'{h:.3f}',
 ha='center', va='bottom', fontsize=9, fontweight='bold', color='#1565C0')
for bar in bars2:
 h = bar.get_height()
 axes[0].text(bar.get_x()+bar.get_width()/2, h+0.01, f'{h:.3f}',
 ha='center', va='bottom', fontsize=9, fontweight='bold', color='#E65100')

# Radar/spider chart (F1 focus)
rf_vals = results.iloc[0][metrics].tolist()
dt_vals = results.iloc[1][metrics].tolist()
N = len(metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
rf_vals += rf_vals[:1]
dt_vals += dt_vals[:1]
angles += angles[:1]
ax = axes[1]
ax.set_facecolor('#F8F9FA')
ax = plt.subplot(1, 2, 2, polar=True)
ax.plot(angles, rf_vals, 'o-', linewidth=2, color='#1565C0', label='Random Forest')
ax.fill(angles, rf_vals, alpha=0.2, color='#1565C0')
ax.plot(angles, dt_vals, 'o-', linewidth=2, color='#E65100', label='Decision Tree')
ax.fill(angles, dt_vals, alpha=0.2, color='#E65100')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Spider Chart Performa Model', fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)

plt.tight_layout()
plt.show()

## 3.4 Kesimpulan & Penyimpanan Model Terbaik

In [ ]:
print('=' * 60)
print(' KESIMPULAN & PENYIMPANAN MODEL TERBAIK')
print('=' * 60)

# Pilih model terbaik berdasarkan F1-Score
# (lebih representatif untuk imbalanced dataset)
if f1_rf >= f1_dt:
 best_model = rf_model
 best_name = 'Random Forest'
 best_acc = acc_rf
 best_prec = prec_rf
 best_rec = rec_rf
 best_f1 = f1_rf
else:
 best_model = dt_model
 best_name = 'Decision Tree'
 best_acc = acc_dt
 best_prec = prec_dt
 best_rec = rec_dt
 best_f1 = f1_dt

print(f' Model Terbaik : {best_name}')
print(f' Akurasi : {best_acc:.4f} ({best_acc*100:.2f}%)')
print(f' Presisi : {best_prec:.4f}')
print(f' Recall : {best_rec:.4f}')
print(f' F1-Score : {best_f1:.4f}')
print()
print(' Alasan Pemilihan:')
print(' F1-Score dipilih sebagai metrik utama karena dataset ini')
print(' IMBALANCED (kelas failure hanya ~3.39%). F1-Score')
print(' memberikan keseimbangan antara Precision dan Recall,')
print(' lebih tepat untuk mengevaluasi kelas minoritas.')

# Simpan model terbaik
joblib.dump(best_model, 'bestmodel.pkl')
print(f'\n Model terbaik ({best_name}) disimpan sebagai bestmodel.pkl')

# Cara load kembali model
print('\n Cara menggunakan model:')
print(' import joblib')
print(" model = joblib.load('bestmodel.pkl')")
print(' prediction = model.predict(X_new)')

## 3.5 Feature Importance (Model Terbaik)

In [ ]:
# Feature importance dari model terbaik
if hasattr(best_model, 'feature_importances_'):
 importances = pd.Series(best_model.feature_importances_, index=feature_cols)
 importances_sorted = importances.sort_values(ascending=True)

 plt.figure(figsize=(10, 6))
 colors_imp = ['#1976D2' if v == importances_sorted.max()
 else '#64B5F6' for v in importances_sorted.values]
 bars = plt.barh(importances_sorted.index, importances_sorted.values,
 color=colors_imp, edgecolor='white')
 plt.xlabel('Importance Score', fontsize=12)
 plt.title(f'Feature Importance — {best_name}',
 fontsize=14, fontweight='bold')
 for bar in bars:
 w = bar.get_width()
 plt.text(w + 0.002, bar.get_y() + bar.get_height()/2,
 f'{w:.4f}', va='center', fontsize=9)
 plt.tight_layout()
 plt.show()
 print('\nTop 3 fitur terpenting:')
 print(importances.sort_values(ascending=False).head(3))

---
## Ringkasan Proyek

| Tahap | Keterangan | Status |
|---|---|---|
| **Step 1** | EDA lengkap (missing value, duplikat, outlier IQR, feature engineering, normalisasi, visualisasi) | |
| **Step 2** | 2 model supervised learning (Random Forest & Decision Tree), split 80:20 + SMOTE | |
| **Step 3** | Evaluasi akurasi, presisi, recall, F1-score, confusion matrix, perbandingan model | |
| **Deploy** | Model terbaik disimpan sebagai `bestmodel.pkl` | |

> **Dataset:** AI4I 2020 Predictive Maintenance | 10.000 baris × 14 kolom
> **Task:** Binary Classification (Machine Failure Detection)